In [1]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep
df = pd.read_csv(r"Downloads/taiv_campaign_measurement_synthetic.csv")

df.head()

,record_id,campaign_id,campaign_name,audience_segment,region,customer_type,treatment_group,reach_pct,frequency,impressions,spend_cad,brand_awareness,purchase_intent,website_visit,store_visit,conversion
0,1,NSB-02,Summer Value,18-24,Quebec,New,Treatment,63.4,1.63,113,12.27,0.4853,0.2271,0,0,0
1,2,NSB-04,Family Meals,18-24,Alberta,Existing,Treatment,50.3,1.87,99,25.54,0.4238,0.2324,0,1,0
2,3,NSB-03,Late Night,35-44,Ontario,New,Control,91.0,1.99,224,76.91,0.4654,0.2490,0,0,0
3,4,NSB-03,Late Night,45-54,Manitoba,Existing,Control,60.6,3.13,224,37.37,0.4698,0.1882,0,0,0
4,5,NSB-01,Spring Launch,25-34,Alberta,Existing,Control,80.0,3.64,368,95.71,0.3830,0.2587,0,0,0


In [2]:
balance_vars = ["audience_segment", "region", "customer_type"]

balance_tables = {}

for var in balance_vars:
    table = pd.crosstab(
        df[var],
        df["treatment_group"],
        normalize="columns"
    ) * 100
    
    balance_tables[var] = table
    
    print(f"\n{var.upper()}")
    print(table.round(2))


AUDIENCE_SEGMENT
treatment_group   Control  Treatment
audience_segment                    
18-24               20.40      19.96
25-34               32.58      32.06
35-44               27.36      28.37
45-54               19.66      19.61

REGION
treatment_group   Control  Treatment
region                              
Alberta             21.74      22.39
British Columbia    18.23      17.88
Manitoba            10.28      10.29
Ontario             29.59      29.55
Quebec              20.16      19.89

CUSTOMER_TYPE
treatment_group  Control  Treatment
customer_type                      
Existing           44.92      45.14
New                55.08      54.86


In [3]:
df.groupby("treatment_group")["conversion"].mean()

treatment_group
Control      0.036002
Treatment    0.044897
Name: conversion, dtype: float64

In [4]:
treatment_rate = df.loc[df["treatment_group"] == "Treatment", "conversion"].mean()
control_rate = df.loc[df["treatment_group"] == "Control", "conversion"].mean()

absolute_difference = treatment_rate - control_rate
print(absolute_difference)

0.008894742299309671


In [5]:
metrics = [
    "conversion",
    "website_visit",
    "store_visit",
    "purchase_intent",
    "brand_awareness"
]

summary = df.groupby("treatment_group")[metrics].mean().T
summary

treatment_group,Control,Treatment
conversion,0.036002,0.044897
website_visit,0.099625,0.115930
store_visit,0.067374,0.080975
purchase_intent,0.237474,0.278986
brand_awareness,0.437997,0.492841


In [6]:
summary["lift_pct"] = (
    (summary["Treatment"] - summary["Control"])
    / summary["Control"]
    * 100
)
print(summary["lift_pct"])

conversion         24.706084
website_visit      16.367007
store_visit        20.186332
purchase_intent    17.480717
brand_awareness    12.521640
Name: lift_pct, dtype: float64


In [7]:
treatment = df.loc[df["treatment_group"] == "Treatment", "conversion"]
control = df.loc[df["treatment_group"] == "Control", "conversion"]

counts = [treatment.sum(), control.sum()]
nobs = [len(treatment), len(control)]

stat, p_value = proportions_ztest(counts, nobs)

print("z-statistic:", stat)
print("p-value:", p_value)

z-statistic: 3.5696918221622425
p-value: 0.00035740143213005757


In [8]:
treatment = df.loc[df["treatment_group"] == "Treatment", "conversion"]
control = df.loc[df["treatment_group"] == "Control", "conversion"]

count_t = treatment.sum()
count_c = control.sum()
n_t = len(treatment)
n_c = len(control)

low, high = confint_proportions_2indep(
    count_t, n_t, count_c, n_c,
    method="wald",
    compare="diff"
)

print("Treatment rate:", count_t / n_t)
print("Control rate:", count_c / n_c)
print("Difference:", (count_t / n_t) - (count_c / n_c))
print("95% CI:", (low, high))

Treatment rate: 0.04489697747133809
Control rate: 0.03600223517202842
Difference: 0.008894742299309671
95% CI: (0.004011155968732965, 0.013778328629886379)


In [9]:
campaign_summary = (
    df.groupby(["campaign_id", "campaign_name", "treatment_group"])["conversion"]
      .mean()
      .unstack()
)

campaign_summary["absolute_difference"] = (
    campaign_summary["Treatment"] - campaign_summary["Control"]
)

campaign_summary["lift_pct"] = (
    campaign_summary["absolute_difference"]
    / campaign_summary["Control"]
    * 100
)

campaign_summary.sort_values("lift_pct", ascending=False)

,treatment_group,Control,Treatment,absolute_difference,lift_pct
campaign_id,campaign_name,,,,
NSB-03,Late Night,0.033438,0.050400,0.016962,50.726668
NSB-01,Spring Launch,0.035201,0.046241,0.011040,31.363357
NSB-04,Family Meals,0.037450,0.044339,0.006888,18.392550
NSB-02,Summer Value,0.037752,0.039198,0.001446,3.830042


In [10]:
combo_counts = (
    df.groupby(
        ["campaign_name", "audience_segment", "treatment_group"]
    )
    .size()
    .unstack(fill_value=0)
)

combo_counts["total"] = combo_counts.sum(axis=1)

combo_counts.sort_values("total", ascending=False)

treatment_group                 Control  Treatment  total
campaign_name audience_segment                           
Summer Value  25-34                1136       1090   2226
Spring Launch 25-34                1069       1131   2200
              35-44                 948        987   1935
Summer Value  35-44                 961        918   1879
Family Meals  25-34                 959        868   1827
Late Night    25-34                 917        910   1827
              35-44                 786        860   1646
Family Meals  35-44                 732        774   1506
Spring Launch 18-24                 704        713   1417
              45-54                 688        694   1382
Summer Value  45-54                 678        680   1358
              18-24                 695        654   1349
Late Night    18-24                 598        578   1176
Family Meals  18-24                 559        544   1103
Late Night    45-54                 570        529   1099
Family Meals  45-54                 527        543   1070

In [11]:
results = []

for (campaign, audience), group in df.groupby(
    ["campaign_name", "audience_segment"]
):
    treatment = group[
        group["treatment_group"] == "Treatment"
    ]["conversion"]

    control = group[
        group["treatment_group"] == "Control"
    ]["conversion"]

    count_t = treatment.sum()
    count_c = control.sum()
    n_t = len(treatment)
    n_c = len(control)

    treatment_rate = count_t / n_t
    control_rate = count_c / n_c

    difference = treatment_rate - control_rate
    lift = difference / control_rate * 100

    z_stat, p_value = proportions_ztest(
        [count_t, count_c],
        [n_t, n_c]
    )

    ci_low, ci_high = confint_proportions_2indep(
        count_t, n_t,
        count_c, n_c,
        method="wald",
        compare="diff"
    )

    results.append({
        "campaign": campaign,
        "audience": audience,
        "treatment_rate": treatment_rate,
        "control_rate": control_rate,
        "difference_pp": difference * 100,
        "lift_pct": lift,
        "p_value": p_value,
        "ci_low_pp": ci_low * 100,
        "ci_high_pp": ci_high * 100,
        "treatment_n": n_t,
        "control_n": n_c
    })

combo_results = pd.DataFrame(results)

combo_results.sort_values("lift_pct", ascending=False)

,campaign,audience,treatment_rate,control_rate,difference_pp,lift_pct,p_value,ci_low_pp,ci_high_pp,treatment_n,control_n
6,Late Night,35-44,0.058140,0.026718,3.142198,117.607973,0.001723,1.214274,5.070122,860,786
12,Summer Value,18-24,0.041284,0.021583,1.970167,91.284404,0.037322,0.101468,3.838866,654,695
10,Spring Launch,35-44,0.059777,0.034810,2.496698,71.723312,0.009881,0.612833,4.380563,987,948
4,Late Night,18-24,0.046713,0.030100,1.661247,55.190311,0.137645,-0.537608,3.860102,578,598
1,Family Meals,25-34,0.050691,0.033368,1.732315,51.915323,0.064041,-0.117473,3.582103,868,959
5,Late Night,25-34,0.058242,0.042530,1.571177,36.942801,0.124656,-0.434138,3.576492,910,917
9,Spring Launch,25-34,0.051282,0.040225,1.105754,27.489565,0.215502,-0.637750,2.849258,1131,1069
8,Spring Launch,18-24,0.030856,0.025568,0.528736,20.679445,0.547948,-1.194809,2.252281,713,704
2,Family Meals,35-44,0.051680,0.043716,0.796374,18.217054,0.469090,-1.354495,2.947243,774,732
3,Family Meals,45-54,0.042357,0.037951,0.440661,11.611418,0.713672,-1.911146,2.792468,543,527


In [12]:
# Export tables for Power BI

df.to_csv("taiv_raw_data.csv", index=False)

summary.to_csv(
    "taiv_overall_summary.csv"
)

campaign_summary.to_csv(
    "taiv_campaign_summary.csv"
)

combo_counts.to_csv(
    "taiv_campaign_audience_counts.csv"
)

combo_results = pd.DataFrame(results)

combo_results.to_csv(
    "taiv_campaign_audience_summary.csv",
    index=False
)

print("Files exported successfully.")

Files exported successfully.
